# Day 3 · Bonus · SAM 3 tự gán nhãn → import vào CVAT

Notebook chạy **SAM 3** (Meta) bằng câu lệnh chữ cho từng class trong `classes.json`, hậu xử lý theo loại task (semantic / instance / panoptic) rồi xuất **đúng định dạng CVAT export** (`Segmentation mask 1.1` / `COCO 1.0`) để **Upload annotations** vào CVAT local.

**Chuẩn bị:** Runtime → Change runtime type → **T4 GPU**. Có tài khoản Hugging Face đã bấm đồng ý license ở trang `facebook/sam3` và có access token (quyền *read*).

Chạy từng ô từ trên xuống. Kết quả nằm trong `bonus_out/` — **không** chép vào `submissions/` (đó là bài chính).

In [ ]:
# BƯỚC 0 · GPU + thư viện
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "Không thấy GPU: đổi Runtime sang T4 GPU")
!pip -q install -U transformers accelerate pycocotools
try:
    from transformers import Sam3Model, Sam3Processor  # noqa
    print("transformers có SAM 3 ✔")
except ImportError:
    print("Bản transformers chưa có SAM 3 → cài bản mới nhất từ GitHub...")
    !pip -q install -U git+https://github.com/huggingface/transformers
    print("Xong. Chọn Runtime → Restart session rồi chạy lại từ ô này.")

In [ ]:
# BƯỚC 1 · Tải fork của bạn (đã push submissions/ + bonus/)
FORK_URL = "https://github.com/gloryTran04/K5-L2-Day03-TranQuangVinh-2A202603054-Image-Segmentation"
import os, pathlib
REPO = pathlib.Path("/content/day3")
if not REPO.exists():
    !git clone --depth 1 {FORK_URL} {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO); sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "bonus"))
print(sorted(p.name for p in (REPO / "submissions").glob("*.zip")))

In [ ]:
# BƯỚC 2 · Đăng nhập Hugging Face (dán token vào ô hiện ra; KHÔNG ghi token vào notebook)
from huggingface_hub import login
login()

In [ ]:
# BƯỚC 3 · Nạp SAM 3 và thử 1 ảnh
import sam3_autolabel as S
from PIL import Image
runner = S.Sam3Runner()          # tải facebook/sam3 lần đầu ~ vài phút
img = Image.open(REPO / "data/tiers/medium_instance/images/000000373353.jpg").convert("RGB")
dets = runner.segment(img, ["car", "person"])
print(len(dets), "mask;", [(p, round(s, 2)) for p, _, s in dets[:8]])

## Chạy cả 9 task
Prompt cho từng class nằm trong `S.PROMPTS`, ngưỡng tin cậy trong `S.SCORE_THR`. Có thể sửa ở đây rồi chạy lại để thử nghiệm (ghi lại thay đổi vào báo cáo bonus).

In [ ]:
# BƯỚC 4 · Tự gán nhãn toàn bộ (≈ 1–3 phút trên T4)
OUT = REPO / "bonus_out"
report = S.run_all(REPO, OUT, runner=runner)

In [ ]:
# BƯỚC 5 · Xem nhanh kết quả
from IPython.display import display
for f in sorted((OUT / "preview").glob("*.jpg")):
    print(f.name); display(Image.open(f).resize((640, int(640 * Image.open(f).height / Image.open(f).width))))

In [ ]:
# BƯỚC 6 · Kiểm định dạng (giống bài chính) + đo tool đã làm được bao nhiêu so với bài cuối của bạn
!python scripts/inspect_submissions.py --dir bonus_out
!python bonus/compare.py --tool bonus_out --final submissions --json bonus_out/agreement.json

**Đọc kết quả:** `agreement` = mức trùng khớp giữa nhãn tự động của tool và bài nộp cuối (đã kiểm tay), dùng chính metric của lab (mIoU / matched IoU × recall / PQ). Đây là *độ giống*, chưa phải độ đúng.

Khi coach phát ground truth, chạy thêm ô dưới để so **tool vs đáp án** và **bài tay vs đáp án**.

In [ ]:
# BƯỚC 7 (sau khi có ground truth) · tải gói đáp án chính thức vào data/tiers/*/groundtruth rồi chạy:
for t in ["easy_semantic", "medium_instance", "hard_panoptic"]:
    print("== TOOL ==");  !python scoring/score.py {t} bonus_out/{t}.zip --group tiers
    print("== FINAL =="); !python scoring/score.py {t} submissions/{t}.zip --group tiers

In [ ]:
# BƯỚC 8 · Tải kết quả về máy để Upload annotations vào CVAT
import shutil
shutil.make_archive("/content/bonus_out", "zip", OUT)
from google.colab import files
files.download("/content/bonus_out.zip")